# Lab 03 Regex and NLP Tokenization
Ali Zaib | FA23-BAI-004 | BAI7

In [ ]:
%pip install -q -r requirements.txt

In [2]:
import re
from pathlib import Path
import nltk
import spacy

try:
    nltk.data.find("tokenizers/punkt_tab/english/")
except LookupError:
    if not nltk.download("punkt_tab", quiet=True):
        raise RuntimeError("NLTK punkt_tab download failed. Check your internet connection.")
if not spacy.util.is_package("en_core_web_sm"):
    from spacy.cli import download
    download("en_core_web_sm")

data_dir = Path("data")

def read_sample(filename):
    return (data_dir / filename).read_text(encoding="utf-8").strip()

In [3]:
# Task 1
text = read_sample("regex_sample.txt")
print("Input:", text)
print("Digits:", re.findall(r"[0-9]", text))
print("Capitalized words:", re.findall(r"\b[A-Z][a-zA-Z]*\b", text))
print("Exactly four digits:", re.findall(r"(?<![0-9])[0-9]{4}(?![0-9])", text))
print("Words of length 3–6:", re.findall(r"\b[a-zA-Z]{3,6}\b", text))

Input: In 2024, Ali studied Python with Sara for 12 weeks. Model X12345 scored 98 points; AI tools cost $50! We plan another run in 2026.
Digits: ['2', '0', '2', '4', '1', '2', '1', '2', '3', '4', '5', '9', '8', '5', '0', '2', '0', '2', '6']
Capitalized words: ['In', 'Ali', 'Python', 'Sara', 'Model', 'AI', 'We']
Exactly four digits: ['2024', '2026']
Words of length 3–6: ['Ali', 'Python', 'with', 'Sara', 'for', 'weeks', 'Model', 'scored', 'points', 'tools', 'cost', 'plan', 'run']


In [4]:
# Task 2
paragraph = read_sample("extraction_sample.txt")
print("Input:", paragraph)
print("Emails:", re.findall(r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b", paragraph))
print("Phones:", re.findall(r"\b03[0-9]{2}-[0-9]{7}\b", paragraph))
print("Dates:", re.findall(r"\b[0-9]{2}/[0-9]{2}/[0-9]{4}\b", paragraph))
print("Percentages:", re.findall(r"\b[0-9]+(?:\.[0-9]+)?%", paragraph))

Input: Contact shuja.ali@example.com or support@nlp-lab.org. Call 0300-1234567 or 0312-7654321 about order ORD-2026-104. The order was placed on 15/10/2026 and is due on 25/10/2026. Discounts are 45% and 12.5%.
Emails: ['shuja.ali@example.com', 'support@nlp-lab.org']
Phones: ['0300-1234567', '0312-7654321']
Dates: ['15/10/2026', '25/10/2026']
Percentages: ['45%', '12.5%']


In [5]:
# Task 3
paragraph = read_sample("tokenization_sample.txt")
nltk_tokens = nltk.word_tokenize(paragraph)
blank_nlp = spacy.blank("en")
spacy_tokens = [token.text for token in blank_nlp(paragraph)]
print("Input:", paragraph)
print("NLTK:", nltk_tokens)
print("spaCy:", spacy_tokens)
print("NLTK token count:", len(nltk_tokens))
print("spaCy token count:", len(spacy_tokens))
# NLTK splits the email at @; spaCy keeps the email as one token.
# NLTK splits the URL into https, :, and //example.com; spaCy keeps it as one token.

Input: I don't think it's easy to build NLP tools. It's useful to test contractions and punctuation. Email shuja@example.com for help. Visit https://example.com for details.
NLTK: ['I', 'do', "n't", 'think', 'it', "'s", 'easy', 'to', 'build', 'NLP', 'tools', '.', 'It', "'s", 'useful', 'to', 'test', 'contractions', 'and', 'punctuation', '.', 'Email', 'shuja', '@', 'example.com', 'for', 'help', '.', 'Visit', 'https', ':', '//example.com', 'for', 'details', '.']
spaCy: ['I', 'do', "n't", 'think', 'it', "'s", 'easy', 'to', 'build', 'NLP', 'tools', '.', 'It', "'s", 'useful', 'to', 'test', 'contractions', 'and', 'punctuation', '.', 'Email', 'shuja@example.com', 'for', 'help', '.', 'Visit', 'https://example.com', 'for', 'details', '.']
NLTK token count: 35
spaCy token count: 31


In [6]:
# Task 4
sentence = read_sample("attributes_sample.txt")
doc = blank_nlp(sentence)
print("Input:", sentence)
print(f'{"Token":<24} {"is_alpha":<10} {"like_num":<10} {"like_email":<12} is_currency')
for token in doc:
    print(f'{token.text:<24} {str(token.is_alpha):<10} {str(token.like_num):<10} {str(token.like_email):<12} {token.is_currency}')
print("Currency tokens:", [token.text for token in doc if token.is_currency])

Input: Ali paid $50 and Rs. 500 for 2 books; email shuja@example.com for details.
Token                    is_alpha   like_num   like_email   is_currency
Ali                      True       False      False        False
paid                     True       False      False        False
$                        False      False      False        True
50                       False      True       False        False
and                      True       False      False        False
Rs                       True       False      False        False
.                        False      False      False        False
500                      False      True       False        False
for                      True       False      False        False
2                        False      True       False        False
books                    True       False      False        False
;                        False      False      False        False
email                    True       False      False   

In [7]:
# Task 5
try:
    nlp = spacy.load("en_core_web_sm")
except OverflowError:
    # Retain the trained components if the serialized tokenizer is incompatible.
    nlp = spacy.load("en_core_web_sm", exclude=["tokenizer"])
paragraph = read_sample("segmentation_sample.txt")
doc = nlp(paragraph)
sentences = [sentence.text for sentence in doc.sents]
for index, sentence in enumerate(sentences, 1):
    print(f"{index}. {sentence}")
print("Total sentences:", len(sentences))
expected = ["Our teacher Dr. Ahmed explained tokenization today.", "Later, Mr. Ali tested the model.", "The results were useful."]
if sentences == expected:
    print("Observation: Dr. and Mr. did not cause incorrect sentence breaks.")
else:
    print("Observation: Segmentation differs from the three expected sentences; inspect the breaks above.")

1. Our teacher Dr. Ahmed explained tokenization today.
2. Later, Mr. Ali tested the model.
3. The results were useful.
Total sentences: 3
Observation: Dr. and Mr. did not cause incorrect sentence breaks.
